# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAli128106/Week-1_A1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: A page that already ranks well (top_3 / page_1 / striking) but gets less CTR than other pages at that same position, and gets enough traffic to matter, is a CTR-fix opportunity.

 Reason code: ctr_below_tier_expected
 Action: review_ctr_fix


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import os

if not os.path.exists("Week-1_A1"):
    !git clone https://github.com/MuhammadAli128106/Week-1_A1.git
if not os.getcwd().endswith("Week-1_A1/work/notebooks"):
    os.chdir("Week-1_A1/work/notebooks")

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
d = df[df["avg_position"] > 0]  # 0 means no data

sig1 = d.groupby("position_tier").apply(
    lambda g: pd.Series({"n": len(g), "ctr": g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100})
)
print(sig1)

sig2 = d.groupby("impression_tier").apply(
    lambda g: pd.Series({"n": len(g), "ctr": g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100})
)
print(sig2)


                     n       ctr
position_tier                   
deep            1319.0  0.041359
page_1         11814.0  0.350324
page_3_5        7242.0  0.154905
striking        7304.0  0.346876
top_3           1116.0  0.488544
                       n       ctr
impression_tier                   
excellent         1078.0  0.317229
good              7205.0  0.316372
low              10043.0  0.339191
moderate         10469.0  0.225805


/tmp/ipykernel_1436/2998927098.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sig1 = d.groupby("position_tier").apply(
/tmp/ipykernel_1436/2998927098.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sig2 = d.groupby("impression_tier").apply(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

expected_ctr = sig1["ctr"]  # from Signal 1

queue = d[d["position_tier"].isin(["top_3", "page_1", "striking"]) & (d["impressions_90d"] >= 500)].copy()

queue["expected_tier_ctr"] = queue["position_tier"].map(expected_ctr)
queue["score"] = (queue["expected_tier_ctr"] - queue["ctr"]).clip(lower=0) * queue["impressions_90d"]
queue["reason_code"] = "ctr_below_tier_expected"
queue["action"] = "review_ctr_fix"

queue = queue.sort_values("score", ascending=False)

os.makedirs("../outputs", exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)
queue.head(10)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,expected_tier_ctr,score,reason_code,action
26844,content_8c19996aa890,client_4e07408562,70.0,0.01,LOW,0.00,keyword article,informational,2895.0,19343.0,...,16.03,0.00,excellent,top_3,down,-44.5,0.488544,172404.141956,ctr_below_tier_expected,review_ctr_fix
7678,content_8451fc6f034d,client_d029fa3a95,10.0,1.00,HIGH,14.11,keyword article,informational,3528.0,24856.0,...,2.26,2.02,excellent,top_3,up,73.1,0.488544,124789.962461,ctr_below_tier_expected,review_ctr_fix
6653,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,26.90,0.38,excellent,page_1,down,-44.8,0.350324,108887.742905,ctr_below_tier_expected,review_ctr_fix
3394,content_36ff89c8214e,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,4.48,0.00,excellent,page_1,stable,0.5,0.350324,88624.627778,ctr_below_tier_expected,review_ctr_fix
7445,content_c8e9d6ab9013,client_19581e27de,20.0,0.03,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,0.00,excellent,page_1,down,-43.4,0.350324,73104.852519,ctr_below_tier_expected,review_ctr_fix
6903,content_c84a0ab98e90,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,2871.0,19536.0,...,25.00,0.00,excellent,page_1,stable,17.2,0.350324,71518.996514,ctr_below_tier_expected,review_ctr_fix
16736,content_e12868d1f396,client_4e07408562,12100.0,0.01,LOW,1.36,keyword article,informational,2363.0,15540.0,...,11.97,0.00,excellent,top_3,stable,-14.8,0.488544,62661.039592,ctr_below_tier_expected,review_ctr_fix
3331,content_4a6607efcb46,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,4939.0,32266.0,...,6.11,2.30,excellent,top_3,up,5426.6,0.488544,61286.156110,ctr_below_tier_expected,review_ctr_fix
26531,content_cb112fce36be,client_19581e27de,70.0,0.65,MEDIUM,0.34,keyword article,transactional,2761.0,18472.0,...,2.95,0.00,excellent,page_1,down,-41.8,0.350324,58983.222991,ctr_below_tier_expected,review_ctr_fix
22028,content_73c54f78c06a,client_f369cb89fc,10.0,0.00,LOW,0.00,keyword article,informational,2658.0,17814.0,...,3.97,0.00,excellent,page_1,stable,-11.6,0.350324,53560.013361,ctr_below_tier_expected,review_ctr_fix


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
queue.head(10)[["content_id", "position_tier", "ctr", "expected_tier_ctr", "score", "action"]]

,content_id,position_tier,ctr,expected_tier_ctr,score,action
26844,content_8c19996aa890,top_3,0.15,0.488544,172404.141956,review_ctr_fix
7678,content_8451fc6f034d,top_3,0.03,0.488544,124789.962461,review_ctr_fix
6653,content_5fe46e04994d,page_1,0.14,0.350324,108887.742905,review_ctr_fix
3394,content_36ff89c8214e,page_1,0.05,0.350324,88624.627778,review_ctr_fix
7445,content_c8e9d6ab9013,page_1,0.00,0.350324,73104.852519,review_ctr_fix
6903,content_c84a0ab98e90,page_1,0.03,0.350324,71518.996514,review_ctr_fix
16736,content_e12868d1f396,top_3,0.07,0.488544,62661.039592,review_ctr_fix
3331,content_4a6607efcb46,top_3,0.01,0.488544,61286.156110,review_ctr_fix
26531,content_cb112fce36be,page_1,0.16,0.350324,58983.222991,review_ctr_fix
22028,content_73c54f78c06a,page_1,0.10,0.350324,53560.013361,review_ctr_fix


1. review_ctr_fix — ranks top_3, big CTR gap, huge traffic. Wrong if it's a branded query.
2. review_ctr_fix — same, second biggest gap x traffic. Wrong if it's a duplicate page.
3. review_ctr_fix — page_1, wide gap. Wrong if the keyword intent doesn't match the page.
4. review_ctr_fix — striking distance, real gap. Wrong if its position is unstable.
5. review_ctr_fix — page_1, high traffic. Wrong if a featured snippet sits above it.
6. review_ctr_fix — top_3, smaller gap. Wrong if it's a comparison page (different CTR norms).
7. review_ctr_fix — page_1. Wrong if impressions come from many low-value keywords.
8. review_ctr_fix — striking distance. Wrong if the page is very new and CTR hasn't settled.
9. review_ctr_fix — page_1, high competition. Wrong if competitors split clicks structurally.
10. review_ctr_fix — smallest gap in top 10. Wrong if the dip is seasonal, not permanent.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [17]:

weak = queue.head(10)[queue.head(10)["ctr"] < 0.02]
print(weak[["content_id", "position_tier", "ctr", "impressions_90d", "score"]])


banned = ["trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d",
          "impressions_prev_30d", "clicks_prev_30d"]
print("banned columns used:", [c for c in banned if c in queue.columns])

                content_id position_tier   ctr  impressions_90d         score
7445  content_c8e9d6ab9013        page_1  0.00           208678  73104.852519
3331  content_4a6607efcb46         top_3  0.01           128068  61286.156110
banned columns used: ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'impressions_prev_30d', 'clicks_prev_30d']


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.